# CSIRO Biomass EDA and Baseline
This notebook explores the competition data and builds a simple tabular baseline to complement the PyTorch image pipeline in the repo.
# Install and Verify Dependencies
# If running locally, ensure requirements are installed. On Kaggle, many are pre-installed.
import sys, subprocess, importlib, os

def pip_install(pkgs):
    for p in pkgs:
        try:
            importlib.import_module(p.split("==")[0])
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)

pkgs = [
    "numpy", "pandas", "scikit-learn", "lightgbm", "xgboost", "optuna",
    "kaggle", "joblib", "mlflow", "matplotlib", "seaborn",
]
pip_install(pkgs)

import numpy as np, pandas as pd
import sklearn
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("lightgbm:", lgb.__version__)
print("xgboost:", xgb.__version__)
# Configure Kaggle API and Download Competition Data
# Use CLI if not already downloaded. This will place files under data/
from pathlib import Path
repo_root = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
data_dir = repo_root / "data"
raw_dir = data_dir
raw_dir.mkdir(parents=True, exist_ok=True)

print("Data directory:", str(raw_dir))
print("If needed, run the PowerShell script: pwsh scripts/download_data.ps1")
# Load Data and Validate Schema
train_csv = raw_dir / "train.csv"
test_csv = raw_dir / "test.csv"
sample_sub_csv = raw_dir / "sample_submission.csv"

assert train_csv.exists(), f"Missing {train_csv}"
assert test_csv.exists(), f"Missing {test_csv}"
assert sample_sub_csv.exists(), f"Missing {sample_sub_csv}"

df_train = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)
df_sub = pd.read_csv(sample_sub_csv)

print("train shape:", df_train.shape)
print("test shape:", df_test.shape)
print("sample_submission shape:", df_sub.shape)
print("train columns:", df_train.columns.tolist())

# Basic checks
assert {"sample_id","image_path","target_name","target"}.issubset(df_train.columns)
assert {"sample_id","image_path","target_name"}.issubset(df_test.columns)
assert {"sample_id","target"}.issubset(df_sub.columns)
assert df_train["sample_id"].is_unique, "Train sample_id not unique per image"
print("Schema validated.")
# Quick Data Profiling and Null Handling
print(df_train.describe(include="all").transpose().head(20))
missing = df_train.isna().mean().sort_values(ascending=False)
print("Missingness (top 20):\n", missing.head(20))

# Simple imputers for tabular
num_cols = ["Pre_GSHH_NDVI","Height_Ave_cm"]
cat_cols = ["State","Species","Sampling_Date"]

for c in num_cols:
    if c in df_train.columns:
        df_train[c] = df_train[c].fillna(df_train[c].median())
for c in cat_cols:
    if c in df_train.columns:
        df_train[c] = df_train[c].fillna(df_train[c].mode().iloc[0])

print("Applied simple imputations.")
# Pivot long train to wide
TARGETS = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g"]
wide = df_train.pivot_table(index=["sample_id","image_path","Sampling_Date","State","Species","Pre_GSHH_NDVI","Height_Ave_cm"],
                            columns="target_name", values="target", aggfunc="first").reset_index()
for t in TARGETS:
    if t not in wide.columns:
        wide[t] = np.nan
print("Wide shape:", wide.shape)
print("Missing targets per column:")
print(wide[TARGETS].isna().sum())
# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(12,8))
axes = axes.flatten()
for i, t in enumerate(TARGETS):
    sns.histplot(wide[t].dropna(), ax=axes[i], kde=True)
    axes[i].set_title(t)
axes[-1].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12,8))
axes = axes.flatten()
for i, t in enumerate(TARGETS):
    sns.histplot(np.log1p(wide[t].dropna()), ax=axes[i], kde=True)
    axes[i].set_title(f"log1p {t}")
axes[-1].axis('off')
plt.tight_layout(); plt.show()
# Scatter NDVI vs targets
if "Pre_GSHH_NDVI" in wide.columns:
    fig, axes = plt.subplots(2, 3, figsize=(12,8))
    axes = axes.flatten()
    for i, t in enumerate(TARGETS):
        sns.scatterplot(x=wide["Pre_GSHH_NDVI"], y=wide[t], ax=axes[i], s=10)
        axes[i].set_title(f"NDVI vs {t}")
    axes[-1].axis('off')
    plt.tight_layout(); plt.show()
# Inspect sample images
import cv2
paths = wide["image_path"].dropna().head(6).tolist()
imgs = []
for p in paths:
    fp = str(raw_dir / p)
    img = cv2.imread(fp)
    if img is None:
        img = np.zeros((224,224,3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    imgs.append(img)

fig, axes = plt.subplots(2,3, figsize=(10,7))
axes = axes.flatten()
for i, img in enumerate(imgs):
    axes[i].imshow(img)
    axes[i].set_title(paths[i])
    axes[i].axis('off')
plt.tight_layout(); plt.show()
# Feature Engineering Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_features = ["Pre_GSHH_NDVI","Height_Ave_cm"]
cat_features = ["State","Species"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)

print("Preprocess pipeline ready.")
# Train/Validation Split and Reproducibility
from sklearn.model_selection import KFold
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
print("Prepared 5-fold CV.")
# Implement Metrics (RMSE)
import math

def rmse(y_true, y_pred):
    return math.sqrt(np.mean((y_true - y_pred)**2))

# Validate metric
print("RMSE test:", rmse(np.array([0,1,2]), np.array([0,1,2])))
# Baseline Model: LightGBM (one target example: Dry_Total_g)
from lightgbm import LGBMRegressor

wide_clean = wide.dropna(subset=["Dry_Total_g"])  # simple baseline on one target
X = wide_clean[num_features + cat_features]
y = wide_clean["Dry_Total_g"].values

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("lgbm", LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=SEED))
])

# Simple train/val split
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, random_state=SEED)
model.fit(Xtr, ytr)
preds = model.predict(Xval)
print("Baseline RMSE:", rmse(yval, preds))
# Cross-Validation Training with OOF
oof = np.zeros(len(y))
fold_metrics = []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
    ytr, yval = y[tr_idx], y[val_idx]
    m = Pipeline(steps=[
        ("preprocess", preprocess),
        ("lgbm", LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=SEED))
    ])
    m.fit(Xtr, ytr)
    p = m.predict(Xval)
    oof[val_idx] = p
    metric = rmse(yval, p)
    fold_metrics.append(metric)
    print(f"Fold {fold} RMSE: {metric:.4f}")
print("OOF RMSE:", rmse(y, oof))
# Hyperparameter Tuning with Optuna
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
    }
    cv_oof = np.zeros(len(y))
    for tr_idx, val_idx in kf.split(X):
        Xtr, Xval = X.iloc[tr_idx], X.iloc[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]
        m = Pipeline(steps=[
            ("preprocess", preprocess),
            ("lgbm", LGBMRegressor(random_state=SEED, **params))
        ])
        m.fit(Xtr, ytr)
        cv_oof[val_idx] = m.predict(Xval)
    return rmse(y, cv_oof)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)
print("Best params:", study.best_params)
# Model Ensembling
from sklearn.linear_model import Ridge

ridge = Pipeline(steps=[("preprocess", preprocess), ("ridge", Ridge(alpha=1.0, random_state=SEED))])
ridge.fit(Xtr, ytr)
p_ridge = ridge.predict(Xval)

xgbm = Pipeline(steps=[("preprocess", preprocess), ("xgb", xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=SEED))])
xgbm.fit(Xtr, ytr)
p_xgb = xgbm.predict(Xval)

blend = 0.6 * preds + 0.2 * p_ridge + 0.2 * p_xgb
print("Blended RMSE:", rmse(yval, blend))
# Feature Importance and Selection (LightGBM)
# Fit LightGBM standalone to extract feature importances
lgbm = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=SEED)
lgb_model = Pipeline(steps=[("preprocess", preprocess), ("lgbm", lgbm)])
lgb_model.fit(Xtr, ytr)
# Access categorical feature names after one-hot
ct = lgb_model.named_steps["preprocess"]
num_feats = num_features
cat_names = list(ct.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_features))
feature_names = num_feats + cat_names
importances = lgbm.feature_importances_
imp = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
print(imp.head(10))
plt.figure(figsize=(8,6))
sns.barplot(data=imp.head(20), x="importance", y="feature")
plt.tight_layout(); plt.show()
# Submission File Generation and Validation
# For tabular baseline, we will create a dummy prediction aligned to sample_submission.
# The PyTorch pipeline in repo handles image-based predictions.
sub = df_test[["sample_id"]].copy()
sub["target"] = 0.0  # placeholder baseline
sub_path = data_dir / "submissions"
sub_path.mkdir(parents=True, exist_ok=True)
fn = sub_path / "submission_tabular_baseline.csv"
sub.to_csv(fn, index=False)
print("Wrote submission:", str(fn))
# Artifact Saving and Experiment Logging
from datetime import datetime
run_dir = repo_root / "artifacts" / datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, run_dir / "lgbm_pipeline.joblib")
np.save(run_dir / "oof.npy", oof)
print("Saved artifacts to:", str(run_dir))
# Unit Tests (lightweight checks)
# In a real project, set up pytest files. Here, perform quick asserts.
assert (pd.read_csv(fn).columns.tolist() == ["sample_id","target"]) , "Submission schema mismatch"
print("Basic tests passed.")
# VS Code Tasks
# Consider adding .vscode/tasks.json to streamline running training/inference scripts.
print("See repo README for training/inference commands.")